In [1]:
from pathlib import Path
import gc
import os
import sys
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

In [2]:
%load_ext autoreload
%autoreload 2

# drGAT パッケージのパスを追加
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler
from drGAT.myutils import get_all_edges_and_labels

In [3]:
data = 'nci'
res, null_mask, S_d, S_c, S_g, drug_feature, gene_norm_gene, gene_norm_cell, A_cg, A_dg = load_data(data)
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

load nci
Done!


In [7]:
def drGAT_new(
    res_mat,
    null_mask,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    params,
    device,
):
    sampler = NewSampler(
        res,
        null_mask, 
        target_dim, 
        target_index,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
    )

    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )

    return best_val_labels, best_val_prob

In [8]:
method = "GAT"
params = {
    "hidden1": 256,  # 少し小さめに
    "hidden2": 128,
    "hidden3": 64,
    "final_mlp_layers": 2,
    "dropout1": 0.1,  # 最初は小さめに
    "dropout2": 0.1,
    "dropout3": 0.1,
    "epochs": 3,  # 3は少なすぎる
    "heads": 1,  # 1から増やしてみる
    "activation": "relu",
    "optimizer": "Adam",
    "lr": 5e-4,  # 少し下げる
    "weight_decay": 1e-5,
    "scheduler": None,
    "norm_type": "BatchNorm",
    "gnn_layer": method,
    "attention_dropout": 0,
    "T_max": 50,  # schedulerがなめらかに動くように長めに
}
params.update(
    {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
    }
)

In [9]:
n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

target_dim = 0  # Cell
# target_dim = 1  # Drug

samples = res.shape[target_dim]

for target_index in tqdm(range(samples)):
    if target_dim:
        if drug_sum.iloc[target_index] < 10:
            continue
    else:
        if cell_sum.iloc[target_index] < 10:
            continue
            
    epochs = []
    true_data_s = pd.DataFrame()
    predict_data_s = pd.DataFrame()
    true_data, predict_data = drGAT_new(
        res_mat=res,
        null_mask=null_mask.values,
        target_dim=target_dim,
        target_index=target_index,
        S_d=S_d,
        S_c=S_c,
        S_g=S_g,
        A_cg=A_cg,
        A_dg=A_dg,
        params=params,
        device=device,
    )

    true_datas = pd.concat(
        [true_datas, pd.DataFrame(true_data).T], ignore_index=True
    )
    predict_datas = pd.concat(
        [predict_datas, pd.DataFrame(predict_data).T], ignore_index=True
    )

  0%|                                                                            | 0/59 [00:00<?, ?it/s]

Using device: cpu


  2%|█▏                                                                  | 1/59 [00:27<26:20, 27.26s/it]

Best model found at epoch 3
Using device: cpu


  3%|██▎                                                                 | 2/59 [00:54<26:07, 27.49s/it]

...

 17%|███████████▎                                                       | 10/59 [04:39<23:01, 28.19s/it]

Best model found at epoch 3
Using device: cpu


 19%|████████████▍                                                      | 11/59 [05:07<22:35, 28.24s/it]

Best model found at epoch 3
Using device: cpu


 19%|████████████▍                                                      | 11/59 [05:32<24:09, 30.19s/it]

KeyboardInterrupt



In [ ]:
sampler = NewSampler(
    res,
    null_mask, 
    target_dim, 
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
)

In [ ]:
sampler.edge_attr[(torch.isnan(sampler.edge_attr))]

In [ ]:
len(sampler.edge_attr)

In [ ]:
len(sampler.edge_index[0])

In [ ]:
def _get_label_df(data, mask):
    mask = mask.to(bool).cpu().numpy()
    rows, cols = np.where(mask)
    values = data[mask]
    
    return pd.DataFrame({
        "Drug": cols,
        "Cell": rows,
        "Label": values.cpu().numpy()
    }).sort_values(["Drug", "Cell"]).reset_index(drop=True)

In [ ]:
train_data = sampler.train_data
train_mask = sampler.train_mask

train_labels_df = _get_label_df(train_data, train_mask)
test_labels_df = _get_label_df(test_data, test_mask)

In [84]:
(sampler.test_labels_df)['Label'].unique()

array([1., 0.], dtype=float32)

In [85]:
(sampler.train_labels_df)['Label'].unique()

array([0., 1.])